In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
import numpy.ma as ma

### functions

In [2]:
img = np.load('.\\data\\STEM img.npy')
mirror = np.load('.\\data\\mirror img.npy')[36:-36,36:-36]

In [18]:
def compute_ij(step_size, mask_size):
    # Generate positions for updating symmetry image
    positions = []
    for i in range(mask_size):
        for j in range(0, mask_size, step_size):
            positions.append([i, j])
        positions.append([i, mask_size - 1])
    return np.array(positions)

def update_mask(row_ind, col_ind, step_size, mask_size):
    mask = np.ones((mask_size, mask_size), dtype=bool)  # Masking all
    i = max(0, row_ind - 1)
    mask[0:i, :] = False
    j = max(col_ind - step_size, 0)
    mask[row_ind, 0:j] = False
    return mask

def update_box(row_ind, col_ind, box):
    box.set_xy((col_ind, row_ind))
    return box


# Parameter settings
mask_size = mirror.shape[0]
step_size = 150

# Create a mask (initially masking everything)
mask = np.ones_like(mirror, dtype=bool)
masked_data1 = ma.array(mirror, mask=mask)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
ax1, ax2, ax3 = axes.ravel()

im1 = ax1.imshow(img, cmap='viridis')
im2 = ax3.imshow(masked_data1, cmap='viridis', vmin=mirror.min(), vmax=mirror.max())

box = plt.Rectangle((0, 0), width=72, height=72, fill=False, ec='r')
ax1.add_patch(box)

ax1.axis('off')
ax2.axis('off')
ax3.axis('off')

pos1 = ax1.get_position()
pos2 = ax3.get_position()

# Compute positions for the sliding kernel
ijs = compute_ij(step_size, mask_size)

def update(frame):
    # here 'global box' is must
    global box
    i, j = ijs[frame]
    mask1 = update_mask(i, j, step_size, mask_size)
    masked_data1.mask = mask1  # Update mask directly
    im2.set_array(masked_data1)

    box = update_box(i, j, box)
    return [box,im2]

# Create the animation
ani = FuncAnimation(
    fig,
    update,
    frames=len(ijs),
    interval=0.05,  # Interval between frames in milliseconds
    blit=True,  # Blit=False to avoid rendering issues
)

## Save to gif

In [19]:
# Save the animation as a GIF
ani.save(
    "reflectional_animation.gif",
    writer=PillowWriter(fps=2000),
    savefig_kwargs={"transparent": True, "pad_inches": 0},
)

plt.close(fig)